# 1. Setup

Install packages

In [3]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()

login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Dataset retrieval

We use the Lichess UCI dataset: https://huggingface.co/datasets/austindavis/lichess-uci

In [4]:
from datasets import load_dataset, Dataset, DatasetDict

ds = load_dataset("austindavis/lichess-uci", "201505")["train"]

# First cut: split off test set — this is quarantined until the end
train_val = ds.train_test_split(test_size=0.05, seed=42)

# Second cut: split train into train and validation
train_val_split = train_val["train"].train_test_split(test_size=0.05, seed=42)

final_splits = {
    "train": train_val_split["train"],
    "val":   train_val_split["test"],
    "test":  train_val["test"],
}

ds = DatasetDict(final_splits)
ds

DatasetDict({
    train: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript'],
        num_rows: 1929144
    })
    val: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript'],
        num_rows: 101534
    })
    test: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript'],
        num_rows: 106878
    })
})

In [5]:
ds["train"][0]

{'Event': 'Rated Bullet tournament https://lichess.org/tournament/MJHHFtgM',
 'Site': 'PPEvEQ91',
 'White': 'Suninheart',
 'Black': 'aila',
 'Result': '0-1',
 'UTCDate': datetime.date(2015, 5, 14),
 'UTCTime': datetime.time(21, 35, 10),
 'WhiteElo': 0,
 'BlackElo': 0,
 'WhiteRatingDiff': -9.0,
 'BlackRatingDiff': 4.0,
 'ECO': 'D05',
 'Opening': "Queen's Pawn Game, Zukertort Variation",
 'TimeControl': '120+0',
 'Termination': 'Normal',
 'Transcript': 'd2d4 d7d5 e2e3 e7e6 g1f3 g8f6 b1d2 c7c5 b2b3 c5d4 e3d4 b8c6 c1b2 f8e7 a2a3 e8g8 f1d3 a7a6 e1g1 c8d7 f3e5 a8b8 d2f3 c6e5 d4e5 f6e8 c2c4 d5c4 b3c4 f7f6 e5f6 e7f6 d1c2 h7h6 d3h7 g8h8 b2f6 d8f6 h7e4 e8d6 a1d1 d6e4 c2e4 d7c6 e4g4 c6f3 g2f3 f6f3 g4f3 f8f3 g1g2 f3a3 d1d2 a3c3 f1d1 c3c4 d2d8 b8d8 d1d8 h8h7 d8b8 b7b5 b8a8 c4a4 g2f3 b5b4 a8b8 a6a5 f3e3 a4a2 f2f4 a2h2 e3e4 h2h3 e4e5 b4b3 e5e6 a5a4 e6f5 a4a3 f5g4 h3c3 g4f5 a3a2 b8a8 b3b2 a8a2 b2b1Q f5e5 b1a2 f4f5 a2e2 e5f4'}

In [ ]:
RESULT_STRINGS = {"1-0", "0-1", "1/2-1/2", "*"}

def clean_transcript(sample):
    cleaned_transcripts = []
    cleaned_moves = []
    num_moves = []
    
    for transcript in sample["Transcript"]:
        # Remove result string from end if present
        moves = transcript.lower().split()
        if moves and moves[-1] in RESULT_STRINGS:
            moves = moves[:-1]
        cleaned_transcripts.append(" ".join(moves))
        cleaned_moves.append(moves)
        num_moves.append(len(moves))
    
    return {
        "Transcript": cleaned_transcripts,
        "Moves": cleaned_moves,
        "Number of Moves": num_moves
    }

ds = ds.map(clean_transcript, batched=True, num_proc=3, desc="Cleaning transcripts")
ds["train"][0]

{'Event': 'Rated Bullet tournament https://lichess.org/tournament/MJHHFtgM',
 'Site': 'PPEvEQ91',
 'White': 'Suninheart',
 'Black': 'aila',
 'Result': '0-1',
 'UTCDate': datetime.date(2015, 5, 14),
 'UTCTime': datetime.time(21, 35, 10),
 'WhiteElo': 0,
 'BlackElo': 0,
 'WhiteRatingDiff': -9.0,
 'BlackRatingDiff': 4.0,
 'ECO': 'D05',
 'Opening': "Queen's Pawn Game, Zukertort Variation",
 'TimeControl': '120+0',
 'Termination': 'Normal',
 'Transcript': 'd2d4 d7d5 e2e3 e7e6 g1f3 g8f6 b1d2 c7c5 b2b3 c5d4 e3d4 b8c6 c1b2 f8e7 a2a3 e8g8 f1d3 a7a6 e1g1 c8d7 f3e5 a8b8 d2f3 c6e5 d4e5 f6e8 c2c4 d5c4 b3c4 f7f6 e5f6 e7f6 d1c2 h7h6 d3h7 g8h8 b2f6 d8f6 h7e4 e8d6 a1d1 d6e4 c2e4 d7c6 e4g4 c6f3 g2f3 f6f3 g4f3 f8f3 g1g2 f3a3 d1d2 a3c3 f1d1 c3c4 d2d8 b8d8 d1d8 h8h7 d8b8 b7b5 b8a8 c4a4 g2f3 b5b4 a8b8 a6a5 f3e3 a4a2 f2f4 a2h2 e3e4 h2h3 e4e5 b4b3 e5e6 a5a4 e6f5 a4a3 f5g4 h3c3 g4f5 a3a2 b8a8 b3b2 a8a2 b2b1q f5e5 b1a2 f4f5 a2e2 e5f4',
 'Moves': ['d2d4',
  'd7d5',
  'e2e3',
  'e7e6',
  'g1f3',
  'g8f6',
  'b1d2

# Tokenization

## Vocabular-ization

We use the WordLevel Tokenizer since every of our moves (like `e2e4`) need to be recorded in full, as well as the fact that moves are separated by spaces.

In [7]:
from collections import Counter

# Special tokens first, then all moves sorted by frequency
# Sorting by frequency is a minor nicety — most common tokens
# get low IDs, which can matter for some embedding implementations
SPECIAL_TOKENS = [
    "[PAD]",        # 0 - padding (you may not need this with flat binary approach)
    "[START]",      # 1 - beginning of game
    "[CHECKMATE]",  # 2
    "[STALEMATE]",  # 3
    "[DRAW_REP]",   # 4 - threefold/fivefold repetition
    "[DRAW_50]",    # 5 - 50/75 move rule
    "[DRAW_MAT]",   # 6 - insufficient material
    "[RESIGN]",     # 7 - resignation
    "[TIMEOUT]",    # 8 - TIMEOUT
    "[UNK]",        # 9 - unknown/malformed move (safety net)
]

move_counter = Counter()
for moves in ds["train"]["Moves"]:
    move_counter.update(moves)

print(f"Unique moves after cleaning: {len(move_counter)}")
# Should be ~1966, and 1-0/0-1/1/2-1/2 should be gone
print(f"1-0 still in counter: {'1-0' in move_counter}")


# Build vocab: special tokens + all moves by descending frequency
all_moves_sorted = [move for move, _ in move_counter.most_common()]
vocab = SPECIAL_TOKENS + all_moves_sorted

token2id = {token: idx for idx, token in enumerate(vocab)}
id2token = {idx: token for token, idx in token2id.items()}

len(vocab)

Unique moves after cleaning: 1968
1-0 still in counter: False


1978

Build a token-to-id mapper, and inverse. We also save the vocab for convenience.

Token test

In [8]:
# Round-trip test — should print True
test_token = "e2e4"
assert id2token[token2id[test_token]] == test_token
print(f"vocab size: {len(vocab)}")
print(f"e2e4 is token {token2id['e2e4']}")
print(f"token 1 is {id2token[1]}")  # should be [START]
print(f"rarest move: {vocab[-1]} at index {len(vocab)-1}")

vocab size: 1978
e2e4 is token 12
token 1 is [START]
rarest move: d2c1b at index 1977


In [9]:
from collections import Counter

term_result_counter = Counter(
    zip(ds["train"]["Termination"], ds["train"]["Result"])
)

for (term, result), count in term_result_counter.most_common():
    print(f"({term!r}, {result!r}): {count:,}")

('Normal', '1-0'): 639,837
('Normal', '0-1'): 579,087
('Time forfeit', '1-0'): 316,064
('Time forfeit', '0-1'): 305,911
('Normal', '1/2-1/2'): 53,901
('Time forfeit', '1/2-1/2'): 14,579
('Abandoned', '0-1'): 11,455
('Abandoned', '1-0'): 8,212
('Rules infraction', '1-0'): 96
('Abandoned', '1/2-1/2'): 2


In [10]:
# mass tokenize games taking advantage of ds.map() again

from typing import Any

TERM_MAP = {
    ("Normal",           "1-0"):     "[RESIGN]",
    ("Normal",           "0-1"):     "[RESIGN]",
    ("Normal",           "1/2-1/2"): "[DRAW_REP]",
    ("Time forfeit",     "1-0"):     "[TIMEOUT]",
    ("Time forfeit",     "0-1"):     "[TIMEOUT]",
    ("Time forfeit",     "1/2-1/2"): "[TIMEOUT]",
    ("Abandoned",        "1-0"):     "[RESIGN]",
    ("Abandoned",        "0-1"):     "[RESIGN]",
    ("Abandoned",        "1/2-1/2"): "[DRAW_REP]",
    ("Rules infraction", "1-0"):     "[RESIGN]",
}

import numpy as np

def make_tokenize_fn(token2id, TERM_MAP):
    START_ID = token2id["[START]"]
    UNK_ID   = token2id["[UNK]"]
    
    def tokenize_batch(batch):
        all_encoded = []
        
        for moves, termination, result in zip(
            batch["Moves"],
            batch["Termination"],
            batch["Result"]
        ):
            end_token = TERM_MAP.get((termination, result))
            
            if not moves or result == "*" or end_token is None:
                all_encoded.append(None)
                continue
            
            ids = [START_ID]
            for move in moves:
                ids.append(token2id.get(move, UNK_ID))
            ids.append(token2id[end_token])
            
            all_encoded.append(ids)
        
        return {"input_ids": all_encoded}
    
    return tokenize_batch

tokenize_fn = make_tokenize_fn(token2id, TERM_MAP)

ds = ds.map(
    tokenize_fn,
    batched=True,
    batch_size=10_000,
    num_proc=3,       # Kaggle gives you multiple cores
    desc="Tokenizing"
)
ds

Tokenizing (num_proc=3):   0%|          | 0/1929144 [00:00<?, ? examples/s]

Tokenizing (num_proc=3):   0%|          | 0/101534 [00:00<?, ? examples/s]

Tokenizing (num_proc=3):   0%|          | 0/106878 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript', 'Moves', 'Number of Moves', 'input_ids'],
        num_rows: 1929144
    })
    val: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript', 'Moves', 'Number of Moves', 'input_ids'],
        num_rows: 101534
    })
    test: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript', 'Moves', 'Number of Moves', 'input_ids'],
        num_rows: 106878
    })
})

## Tokenizer

We now build the actual tokenizer.

We use the WordLevel Tokenizer since every of our moves (like `e2e4`) need to be recorded in full, as well as the fact that moves are separated by spaces.

In [11]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(WordLevel(vocab=token2id, unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

tokenizer.add_special_tokens(SPECIAL_TOKENS)

tokenizer.save("./tokenizer/tokenizer.json")
print("Tokenizer saved.")

Tokenizer saved.


Tokenizer 

In [12]:
enc = tokenizer.encode("e2e4 e7e5 g1f3 b8c6")
print(enc.tokens)   # ['e2e4', 'e7e5', 'g1f3', 'b8c6']
print(enc.ids)      # [some integers matching your token2id]

# Round trip
decoded = tokenizer.decode(enc.ids)
print(decoded)      # 'e2e4 e7e5 g1f3 b8c6'

['e2e4', 'e7e5', 'g1f3', 'b8c6']
[12, 21, 10, 17]
e2e4 e7e5 g1f3 b8c6


In [13]:
# Encode a game prefix to feed to the model
moves_so_far = "e2e4 e7e5 g1f3"
ids = tokenizer.encode(moves_so_far).ids
# prepend [START] manually since encode() doesn't add it automatically
ids = [token2id["[START]"]] + ids
print(ids)

# Decode model output back to moves
predicted_id = 423  # whatever the model sampled
predicted_move = tokenizer.id_to_token(predicted_id)  # "b8c6"
print(predicted_move)

[1, 12, 21, 10]
b3b2


In [14]:
print("Special Tokens:", [token for token in token2id if token.startswith("[")])

Special Tokens: ['[PAD]', '[START]', '[CHECKMATE]', '[STALEMATE]', '[DRAW_REP]', '[DRAW_50]', '[DRAW_MAT]', '[RESIGN]', '[TIMEOUT]', '[UNK]']


# Model

## Model configuration

In [15]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size          = len(vocab),      # 1980
    n_positions         = 256,             # max sequence length
    n_embd              = 256,             # d_model
    n_layer             = 6,               # transformer blocks
    n_head              = 8,               # attention heads
    n_inner             = 1024,            # MLP hidden dim (4 * n_embd)
    activation_function = "gelu_new",
    resid_pdrop         = 0.1,
    embd_pdrop          = 0.1,
    attn_pdrop          = 0.1,
    bos_token_id        = token2id["[START]"],
    eos_token_id        = [
        token2id["[RESIGN]"],
        token2id["[TIMEOUT]"],
        token2id["[DRAW_REP]"],
        token2id["[CHECKMATE]"],
        token2id["[STALEMATE]"],
        token2id["[DRAW_MAT]"],
        token2id["[DRAW_50]"],
    ],
    attn_implementation = "sdpa",          # the one good part from that line
)

# Randomly initialized — no pretrained weights
model = GPT2LMHeadModel(config)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
# Should be ~8-10M

Parameters: 5,310,976


## Chunking


In [16]:
from datasets import Dataset

BLOCK_SIZE = 256

def chunk_dataset(split: Dataset, block_size=BLOCK_SIZE):
    # Flatten all input_ids across all games into one stream
    all_ids = []
    for ids in split["input_ids"]:
        if ids is not None:
            all_ids.extend(ids)
    
    print(f"Total tokens: {len(all_ids):,}")
    
    # Drop the remainder that doesn't fill a complete block
    n_chunks = len(all_ids) // block_size
    all_ids  = all_ids[:n_chunks * block_size]
    
    # Cut into fixed-length chunks
    chunks = [
        all_ids[i * block_size : (i + 1) * block_size]
        for i in range(n_chunks)
    ]
    
    print(f"Total chunks: {n_chunks:,}")
    return Dataset.from_dict({"input_ids": chunks})

print("Chunking train...")
train_chunked = chunk_dataset(ds["train"])

print("\nChunking val...")
val_chunked = chunk_dataset(ds["val"])

# Test split stays unchunked — used for behavioral evaluation later
print(f"\nTest split kept as-is: {len(ds['test']):,} games")

Chunking train...
Total tokens: 130,647,926
Total chunks: 510,343

Chunking val...
Total tokens: 6,877,487
Total chunks: 26,865

Test split kept as-is: 106,878 games
